In [ ]:
#! Le departement 94 n'as pas d'information météo sur la période recherhé

In [35]:
# import gevent.monkey #* Handle errors between requests and grequests
# gevent.monkey.patch_all()

import pandas as pd
import numpy as np
from pprint import pprint
import time
import os
import matplotlib.pyplot as plt
from typing import *
import math

In [ ]:
#* Constants
startingDate = "2021-01-01"
endingDate = "2023-12-31"

regions_departements = {
    "Auvergne-Rhône-Alpes": [
        "01", "03", "07", "15", "26", "38", "42", "43", "63", "69", "73", "74"
    ],
    "Bourgogne-Franche-Comté": [
        "21", "25", "39", "58", "70", "71", "89", "90"
    ],
    "Bretagne": [
        "22", "29", "35", "56"
    ],
    "Centre-Val de Loire": [
        "18", "28", "36", "37", "41", "45"
    ],
    "Corse": [
        "20"
    ],
    "Grand Est": [
        "08", "10", "51", "52", "54", "55", "57", "67", "68", "88"
    ],
    "Hauts-de-France": [
        "02", "59", "60", "62", "80"
    ],
    "Ile-de-France": [
        "75", "77", "78", "91", "92", "93", "94", "95"
    ],
    "Nouvelle-Aquitaine": [
        "16", "17", "19", "23", "24", "33", "40", "47", "64", "79", "86", "87"
    ],
    "Normandie": [
        "14", "27", "50", "61", "76"
    ],
    "Occitanie": [
        "09", "11", "12", "30", "31", "32", "34", "46", "48", "65", "66", "81", "82"
    ],
    "Pays de la Loire": [
        "44", "49", "53", "72", "85"
    ],
    "Provence-Alpes-Côte d'Azur": [
        "04", "05", "06", "13", "83", "84"
    ]
}


In [5]:
"""
Get the meteo data from the gouv API 
"""

APIurl = "https://tabular-api.data.gouv.fr/api/resources/"



relevantCols = ["NUM_POSTE","AAAAMMJJ","TN","TX","TM","TNSOL","TN50","DG","FF2M","DRR"]
# reponse = requests.get(APIurl+dataIds[0])

# data = pd.DataFrame(reponse.json()['data'])
# data


In [38]:
meteoIds =  []
with  open("ressourcesLink.txt") as fileIds:
    for link in fileIds:
        meteoIds.append(link[-37:-1])
        
nbDep = len(meteoIds)

In [29]:
""" SWAGGER AND PROFILE"""
# swagger = requests.get("https://tabular-api.data.gouv.fr/api/resources/c79aaafe-8017-4d2b-8884-57b5391da5bc/swagger/")
# for l in swagger.iter_lines():
#     print(l)
reponse = requests.get("https://tabular-api.data.gouv.fr/api/resources/c79aaafe-8017-4d2b-8884-57b5391da5bc/profile/")
print(reponse.json()['profile'])


{'header': ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR', 'TN', 'QTN', 'HTN', 'QHTN', 'TX', 'QTX', 'HTX', 'QHTX', 'TM', 'QTM', 'TNTXM', 'QTNTXM', 'TAMPLI', 'QTAMPLI', 'TNSOL', 'QTNSOL', 'TN50', 'QTN50', 'DG', 'QDG', 'FFM', 'QFFM', 'FF2M', 'QFF2M', 'FXY', 'QFXY', 'DXY', 'QDXY', 'HXY', 'QHXY', 'FXI', 'QFXI', 'DXI', 'QDXI', 'HXI', 'QHXI', 'FXI2', 'QFXI2', 'DXI2', 'QDXI2', 'HXI2', 'QHXI2', 'FXI3S', 'QFXI3S', 'DXI3S', 'QDXI3S', 'HXI3S', 'QHXI3S', 'DRR', 'QDRR'], 'columns': {'DG': {'score': 1.0, 'format': 'int', 'python_type': 'int'}, 'RR': {'score': 1.0, 'format': 'float', 'python_type': 'float'}, 'TM': {'score': 1.0, 'format': 'float', 'python_type': 'float'}, 'TN': {'score': 1.0, 'format': 'float', 'python_type': 'float'}, 'TX': {'score': 1.0, 'format': 'float', 'python_type': 'float'}, 'DRR': {'score': 1.0, 'format': 'int', 'python_type': 'int'}, 'DXI': {'score': 1.0, 'format': 'int', 'python_type': 'int'}, 'DXY': {'score': 1.0, 'format': 'int', 'python_type': 

In [ ]:
#* Test call API
query = (
"page_size=50"
"&columns=NUM_POSTE,AAAAMMJJ,TN,TX,TM,TNSOL,TN50,DG,FF2M,DRR"
"&AAAAMMJJ__greater="+startingDate+""
"&AAAAMMJJ__less="+endingDate+""
)
reponse = requests.get("https://tabular-api.data.gouv.fr/api/resources/c79aaafe-8017-4d2b-8884-57b5391da5bc/data/?"+query)
pprint(reponse.json())
if reponse.status_code==200:
    data = pd.DataFrame(reponse.json()['data'])
else:
    pprint(reponse.json())
    
data.head()

{'data': [{'AAAAMMJJ': '2021-01-01',
           'DG': 439,
           'DRR': None,
           'FF2M': None,
           'NUM_POSTE': '01014002',
           'TM': 0.2,
           'TN': -0.1,
           'TN50': None,
           'TNSOL': None,
           'TX': 0.7},
          {'AAAAMMJJ': '2021-01-02',
           'DG': 1096,
           'DRR': None,
           'FF2M': None,
           'NUM_POSTE': '01014002',
           'TM': -0.5,
           'TN': -2.7,
           'TN50': None,
           'TNSOL': None,
           'TX': 0.5},
          {'AAAAMMJJ': '2021-01-03',
           'DG': 1440,
           'DRR': None,
           'FF2M': None,
           'NUM_POSTE': '01014002',
           'TM': -0.8,
           'TN': -1.1,
           'TN50': None,
           'TNSOL': None,
           'TX': -0.1},
          {'AAAAMMJJ': '2021-01-04',
           'DG': 1408,
           'DRR': None,
           'FF2M': None,
           'NUM_POSTE': '01014002',
           'TM': -1.7,
           'TN': -4.1,
           'TN5

,NUM_POSTE,AAAAMMJJ,TN,TX,TM,TNSOL,TN50,DG,FF2M,DRR
0,01014002,2021-01-01,-0.1,0.7,0.2,None,None,439,None,None
1,01014002,2021-01-02,-2.7,0.5,-0.5,None,None,1096,None,None
2,01014002,2021-01-03,-1.1,-0.1,-0.8,None,None,1440,None,None
3,01014002,2021-01-04,-4.1,0.5,-1.7,None,None,1408,None,None
4,01014002,2021-01-05,-2.0,0.3,-0.9,None,None,1244,None,None


In [61]:
def getFileName(step:Literal["raw","cleaned"],dataSource:Literal["meteo","energy"],departementNumber:int=None,region:str=None,year:int=None):
    if step=="raw" and dataSource=="energy":
        # print(os.path.join(step+"Data",dataSource,f"eCO2mix_RTE_{region}_Annuel-Definitif_{year}.zip"))
        return os.path.join(step+"Data",dataSource,f"eCO2mix_RTE_{region}_Annuel-Definitif_{year}.xlsx")
    
    fileNameBase = "DEP-"
    number = str(departementNumber) if departementNumber>10 else "0"+str(departementNumber)
    return os.path.join(step+"Data",dataSource,fileNameBase+number+".csv")

In [ ]:
#* Get all data :
baseAPI = "https://tabular-api.data.gouv.fr/api/resources/"
query = (
        "/data/?"
        "page_size=200"
        "&columns="+",".join(relevantCols)+""
        "&AAAAMMJJ__greater=2021-01-01" 
        "&AAAAMMJJ__less=2025-01-01"
    )

#* ~3min per departement
def getDepMeteoData(id:str):
    import requests

    isNext = True
    
    data = pd.DataFrame()
    totalRequest = baseAPI+id+query

    #* Loop throw all pages
    while isNext:
        reponse = requests.get(totalRequest)

        if reponse.status_code==200:
            reponse = reponse.json()

            if data.shape[0]>0:
                data = pd.concat([data,pd.DataFrame(reponse['data'])]) #* Concat all data together
            else:
                data = pd.DataFrame(reponse['data'])

            if reponse["links"]["next"]:
                totalRequest = reponse["links"]["next"]
                time.sleep(0.01) #* The API limit rate is 100/seconds
            else: #* End of the data
                isNext = False

            print(f"\rPage : {reponse['meta']['page']} / {reponse['meta']['total']/reponse['meta']['page_size']} ",end='', flush=True)

        else:
            print('ERROR DURING REQUEST',reponse.json())
            break
    print('')
    return data


def getMeteoData():
    for dep,meteoId in list(enumerate(meteoIds))[0:1]:
        print(f"Département {dep} / {len(meteoIds)}")
        meteoData = getDepMeteoData(meteoId)
        print("data size ",meteoData.shape)
        # time.sleep(5)
        # meteoData.to_csv(getFileName("raw","meteo",dep))

getMeteoData()

Département 0 / 99
Page : 20 / 120.97 

C:\Users\felib\AppData\Local\Temp\ipykernel_2684\1551407836.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data,pd.DataFrame(reponse['data'])]) #* Concat all data together


Page : 121 / 120.97 
data size  (24194, 10)


C:\Users\felib\AppData\Roaming\Python\Python313\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [ ]:
#* -------------- ASYNCRHONOUS --------------------
#* Get all data :
apiPageSizeLimit =200
baseAPI = "https://tabular-api.data.gouv.fr/api/resources/"
query = (
        "/data/?"
        "page_size="+str(apiPageSizeLimit)+""
        "&columns="+",".join(relevantCols)+""
        "&AAAAMMJJ__greater=2021-01-01" 
        "&AAAAMMJJ__less=2025-01-01"
    )
data = pd.DataFrame()

def handleApiResult(reponse):
    global data
    # print("API HANDLER")
    if reponse.status_code==200:
        reponse = reponse.json()
        print(f"\rPage : {reponse['meta']['page']} / {reponse['meta']['total']/reponse['meta']['page_size']} ",end='', flush=True)
        data = pd.concat([data,pd.DataFrame(reponse['data'])]) #* Concat all data together

    else:
        print('ERROR DURING REQUEST',reponse.json())

#* ~3min per departement
def getDepMeteoDataAsync(dep,id:str):
    global data
    import grequests
    
    totalRequest = baseAPI+id+query
    #* first request : 
    req = grequests.get(totalRequest)
    reponse = grequests.map([req])[0]
    if reponse.status_code==200:
        reponse = reponse.json()

        data = pd.DataFrame(reponse['data'])
        nbPages = math.floor(reponse["meta"]["total"]/reponse["meta"]["page_size"]) #* We already get the first page
        print(f"First call done, Calls to make :{nbPages} ")
        #* Loop throw all pages
        pendingCall = []
        page = 2
        while page < nbPages+1:
            currentRequest = totalRequest + "&page="+str(page)
            pendingCall.append(grequests.get(currentRequest)) 
            if page%100==0: #* We make call by batch of 100
                results = grequests.map(pendingCall)
                print("")
                print(f"Results call for dep {dep}: {results}")
                for result in results:
                    handleApiResult(result)
                time.sleep(1) #* The API limit rate is 100/seconds
                pendingCall = []
                # break
            print('\rnb url to call ',len(pendingCall),end='',flush=True)

            page +=1

        results = grequests.map(pendingCall)
        print(f"Results call for dep {dep}: {results}")  

    else:
        print('ERROR DURING REQUEST',reponse)

    print('')
    return data


def getMeteoData():
    for dep,meteoId in list(enumerate(meteoIds)):
        print(f"Département {dep} / {len(meteoIds)}")
        meteoData = getDepMeteoDataAsync(dep+1,meteoId)
        print(meteoData.head())
        print("data size ",meteoData.shape)
        # time.sleep(5)
        meteoData.to_csv(getFileName("raw","meteo",dep+1))
getMeteoData()

Département 94 / 95
ERROR DURING REQUEST <Response [400]>

Empty DataFrame
Columns: []
Index: []
data size  (0, 0)


C:\Users\felib\AppData\Roaming\Python\Python313\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [40]:
"""TRANSFORM DATA """

for dep in range(1,nbDep+1):
    rawData = pd.read_csv(getFileName("raw","meteo",dep-1),index_col=0)

    cleanData = rawData.drop(['NUM_POSTE'],axis=1)
    cleanData.groupby(['AAAAMMJJ']).mean() #* Compute the mean by day on differents stations

    cleanData.to_csv(getFileName("cleaned","meteo",dep))

# plt.plot(cleanData["AAAAMMJJ"],cleanData["TN"])


C:\Users\felib\AppData\Roaming\Python\Python313\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [ ]:
"""====================  ENERGY  ===================="""

#* Filtering and agreging the data
# for region in regions_departements.keys():
for region in ["Auvergne-Rhône-Alpes"]:

    data21 = pd.read_excel(getFileName('raw',"energy",region=region,year=2021),engine="openpyxl")
    data22 = pd.read_excel(getFileName('raw',"energy",region=region,year=2022),engine="openpyxl")
    data23 = pd.read_excel(getFileName('raw',"energy",region=region,year=2023),engine="openpyxl")
    print('files opened')
    totalData = pd.concat([data21,data22,data23])
    totalData = totalData[["Périmètre","Date","Consommation"]]
    print(totalData.shape)
    print(totalData.shape,totalData.head())
    
    totalData.groupby(["Date"])
    print(totalData.shape,totalData.head())
    # totalData.to_csv(getFileName("cleaned","energy",region=region))

files opened
(105123, 3)               Périmètre       Date  Consommation
0  Auvergne-Rhône-Alpes 2021-01-01        8586.0
1  Auvergne-Rhône-Alpes 2021-01-01           NaN
2  Auvergne-Rhône-Alpes 2021-01-01        8373.0
3  Auvergne-Rhône-Alpes 2021-01-01           NaN
4  Auvergne-Rhône-Alpes 2021-01-01        8134.0
(105123, 3)               Périmètre       Date  Consommation
0  Auvergne-Rhône-Alpes 2021-01-01        8586.0
1  Auvergne-Rhône-Alpes 2021-01-01           NaN
2  Auvergne-Rhône-Alpes 2021-01-01        8373.0
3  Auvergne-Rhône-Alpes 2021-01-01           NaN
4  Auvergne-Rhône-Alpes 2021-01-01        8134.0


C:\Users\felib\AppData\Roaming\Python\Python313\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:
